In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path

print(Path.cwd())
RAW = Path("C:/Users/danya/OneDrive/Desktop/Smart_Electricity_Demand_Anomaly_Detection_System/Data/raw/household_power_consumption.txt")
df = pd.read_csv(
    RAW,
    sep = ";",
    na_values = ["?"],
    low_memory = False
)

print(df.head())
print(df.shape)
print(df.info())

In [ ]:
df["DateTime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst = True
)

df = df.drop(columns = ["Date", "Time"])
df = df.sort_values("DateTime")
df = df.set_index("DateTime")

print(df.head())
print(df.tail())
print(df.index.is_monotonic_increasing)
print(df.index.to_series().diff().value_counts())

In [ ]:
print(df.dtypes)

In [ ]:
print("Null values in each column:\n", df.isnull().sum())
print("Percentage: \n", (df.isnull().sum() / len(df)) * 100)
print("Null values in index:", df.index.isnull().sum())
missing_per_row = df.isna().sum(axis=1)
print("Missing per row: \n", df.isna().sum(axis=1).value_counts())

is_missing = df["Global_active_power"].isna()
gap_id = (is_missing != is_missing.shift()).cumsum()
gap_len = is_missing.groupby(gap_id).transform("sum")

gap_lengths = is_missing[is_missing].groupby(gap_id[is_missing]).size()
print("Number of gaps:", len(gap_lengths))
print(gap_lengths.describe())
print(gap_lengths.sort_values(ascending=False).head(10))

short = is_missing & (gap_len <= 60)

interp = df.interpolate(method="time", limit_area="inside")
df.loc[short] = interp.loc[short]

print("Filled rows:", short.sum())
print("Still missing:\n", df.isnull().sum())

In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5)*(IQR)
upper_bound = Q3 + (1.5)*(IQR)

outliers = (
    (df < lower_bound) |
    (df > upper_bound)
)
print(outliers.sum())
print("Standard Deviation: \n", df.std())

In [ ]:
print(df.describe())
print(df.info())
print((df < 0).sum())

constant_columns = [
    col for col in df.columns
        if df[col].nunique() <= 1 
]
print("constant columns: \n", constant_columns)

corr_matrix = df.corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k = 1).astype(bool)
)
high_corr = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.95)
]
print("Highly correlated columns: \n", high_corr)

from statsmodels.tsa.stattools import adfuller

for col in df.columns:
    s = df[col].resample("h").mean().dropna()
    print(col, adfuller(s)[:2])

In [ ]:
import matplotlib.dates as mdates
plt.figure(figsize = (15, 5))
plt.plot(
    df.index[:1000],
    df["Global_active_power"].iloc[:1000]
)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xlabel("Time (Hrs)")
plt.ylabel("Active Power (Kw)")
plt.title("Power Consumption")
plt.show()

df["hour"] = df.index.hour
hourly_consumption = df.groupby("hour")[
    "Global_active_power"
].mean()

hourly_consumption.plot(
    kind = "line",
    figsize = (12, 5)
)
plt.title("Average electricty consumption by hour")
plt.ylabel("Power Consumption (Kw)")
plt.xlabel("Hour of the day")
plt.show()

plt.figure(figsize = (8, 3))
sns.boxplot(x = df["Global_active_power"])
plt.title("Global Active Power: Outlier Detection")
plt.xlabel("Global Active Power (KW)")
plt.show()

plt.figure(figsize = (8, 3))
sns.boxplot(x = df["Voltage"])
plt.title("Voltage Outlier Detection")
plt.xlabel("Voltage")
plt.show()

df.hist(figsize=(10, 4), bins = 30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(),
    annot = True,
    cmap = "coolwarm",
    fmt = ".2f"
)
plt.show()

df.plot(
    subplots = True,
    figsize = (12, 8),
    legend = True
)
plt.tight_layout()
plt.show()

In [ ]:

df.to_parquet("C:/Users/danya/OneDrive/Desktop/Smart_Electricity_Demand_Anomaly_Detection_System/Data/processed/Cleaned.parquet")   
